In [5]:
from Bio.PDB import PDBParser, PDBIO
import numpy as np
import pandas as pd
def extract_plddt_scores_residuewise(pdb_file):
    """
    Extracts residue-wise PLDDT scores by averaging the B-factor values for each residue in a PDB file.

    Args:
        pdb_file (str): Path to the PDB file.

    Returns:
        list: A list of average PLDDT scores for each residue.
    """
    residue_scores = {}
    residue_atom_counts = {}

    try:
        with open(pdb_file, 'r') as file:
            for line in file:
                if line.startswith("ATOM"):  # Process only ATOM lines
                    try:
                        # Extract chain ID, residue number, and B-factor
                        chain_id = line[21].strip()  # Chain ID (column 22)
                        residue_number = int(line[22:26].strip())  # Residue number (columns 23-26)
                        b_factor = float(line[60:66].strip())  # B-factor (columns 61-66)

                        # Use (chain_id, residue_number) as key
                        key = (chain_id, residue_number)

                        # Initialize or update the cumulative score and atom count
                        if key not in residue_scores:
                            residue_scores[key] = 0.0
                            residue_atom_counts[key] = 0

                        residue_scores[key] += b_factor
                        residue_atom_counts[key] += 1
                    except ValueError:
                        print(f"Skipping invalid line: {line.strip()}")
                        continue

        # Calculate average PLDDT for each residue
        residue_avg_scores = [
            residue_scores[(chain, residue)] / residue_atom_counts[(chain, residue)]
            for (chain, residue) in residue_scores
        ]

        return residue_avg_scores
    except FileNotFoundError:
        print(f"PDB file not found: {pdb_file}")
        return []
    except Exception as e:
        print(f"Error reading PDB file {pdb_file}: {e}")
        return []



def extract_disordered_rsa(protein_id, rsa_file, disorder_file):
    """
    Extracts Relative Solvent Accessibility (RSA) and disorder scores for a given protein ID.

    Args:
        protein_id (str): The protein ID to extract data for.
        rsa_file (str): Path to the RSA data file (TSV format).
        disorder_file (str): Path to the disorder prediction file.

    Returns:
        tuple: A tuple containing RSA values and disorder scores as numpy arrays.
    """
    print(f"Extracting disordered regions and RSA for: {protein_id}")

    # Load RSA data
    try:
        df_rsa = pd.read_csv(rsa_file, sep='\t')
        df_rsa = df_rsa[df_rsa['name'].str.startswith(protein_id)]
        rsa_values = df_rsa["rsa"].values if not df_rsa.empty else np.array([])
    except Exception as e:
        print(f"Error reading RSA file: {e}")
        rsa_values = np.array([])

    # Load disorder prediction data
    data = []
    try:
        with open(disorder_file, 'r') as file:
            current_protein_id = None
            for line in file:
                if line.startswith(">"):
                    current_protein_id = line.strip()[1:]
                else:
                    parts = line.strip().split("\t")
                    if len(parts) == 4 and current_protein_id:
                        position, amino_acid, score, indicator = parts
                        data.append({
                            "Protein_ID": current_protein_id,
                            "Position": int(position),
                            "Amino_Acid": amino_acid,
                            "Score": float(score),
                            "Indicator": int(indicator)
                        })

        df_disorder = pd.DataFrame(data)
        filtered_disorder = df_disorder[df_disorder["Protein_ID"] == protein_id.replace("-", "")]
        disorder_scores = filtered_disorder["Score"].values if not filtered_disorder.empty else np.array([])
    except Exception as e:
        print(f"Error reading disorder file: {e}")
        disorder_scores = np.array([])

    return rsa_values, disorder_scores

# List of proteins to process
proteins = [
    "CG14125-PA", "Muc91C-PA", "fon-PB", "CG42525-PA", "Muc91C-PB",
    "fon-PC", "CG42525-PB", "Mur18B-PA", "psd-PA", "Cp7Fc-PA",
    "Mur29B-PA", "resilin-PA", "Muc11A-PB", "Mur89F-PC", "resilin-PB",
    "Muc68Ca-PA", "Ppn-PC", "tnc-PC", "Muc68D-PB", "Sgs1-PA",
    "tnc-PD", "Muc68E-PA", "TwdlC-PA"
]

# File paths
rsa_file_path = "./AlphaFoldDIsorder/output_pred.tsv"
disorder_file_path = "./AlphaFoldDIsorder/merged_output.txt"
pdb_base_path = "./AlphaFoldDIsorder/pdb_symfiles/"

# Collect results for CSV
results = []

for protein_id in proteins:
    rsa, disorder = extract_disordered_rsa(protein_id, rsa_file_path, disorder_file_path)
    pdb_file = f"{pdb_base_path}{protein_id}_unrelaxed_rank_001.pdb"
    plddt_scores = extract_plddt_scores_residuewise(pdb_file)
    print(plddt_scores)
    plddt_scores = np.array(plddt_scores)
    rsa_corr, disorder_corr, rsa_disorder_corr = None, None, None

    if rsa.size > 0 and plddt_scores.size > 0:
        rsa_corr = np.corrcoef(rsa, plddt_scores[:len(rsa)])[0, 1]
        print(rsa_corr)
    if disorder.size > 0 and plddt_scores.size > 0:
        disorder_corr = np.corrcoef(disorder, plddt_scores[:len(disorder)])[0, 1]

    if rsa.size > 0 and disorder.size > 0:
        rsa_disorder_corr = np.corrcoef(rsa, disorder)[0, 1]

    results.append({
        "Protein_ID": protein_id,
        "RSA_PLDDT_Correlation": rsa_corr,
        "Disorder_PLDDT_Correlation": disorder_corr,
        "RSA_Disorder_Correlation": rsa_disorder_corr
    })

# Save results to a CSV file
output_file = "correlation_results.csv"
df_results = pd.DataFrame(results)
df_results.to_csv(output_file, index=False)

print(f"Results saved to {output_file}")


Extracting disordered regions and RSA for: CG14125-PA
[81.69, 87.12, 86.56, 92.0, 92.55999999999997, 94.19, 93.62, 93.69, 94.62, 95.81, 95.44, 94.31, 94.25, 93.06, 91.38, 88.06, 88.06000000000002, 83.75, 78.62, 75.75, 67.62, 65.94000000000001, 65.12, 65.12, 65.12, 69.19, 62.59000000000002, 65.25, 66.19, 62.15999999999999, 67.0, 66.56, 72.06, 70.69, 62.25, 63.40999999999998, 65.25, 58.15999999999998, 53.88, 60.0, 55.90999999999998, 52.22000000000001, 59.02999999999999, 57.0, 54.940000000000005, 55.72000000000001, 53.65999999999998, 54.65999999999998, 48.62, 51.38, 46.03, 51.06, 50.5, 60.15999999999998, 52.88, 53.470000000000006, 50.88, 45.25, 53.5, 51.52999999999999, 58.34000000000002, 52.84000000000002, 59.340000000000025, 56.09000000000002, 50.28000000000001, 48.71999999999999, 49.12, 55.75, 49.19, 52.19, 54.22000000000001, 56.77999999999999, 48.06, 43.28000000000001, 45.53, 47.21999999999999, 53.69, 46.0, 42.22, 52.47000000000001, 41.12, 42.66, 35.78, 51.0, 46.84, 51.94, 48.91, 54.19

AttributeError: 'list' object has no attribute 'size'